# Appendix E: Singularities and Intersections

    **Source orientation:** McDuff-Salamon, *J-holomorphic Curves and Symplectic Topology*, Appendix E, printed pp. 653-694; PDF pp. 668-709. Sections: E.1-E.7.

    ## Chapter Goal

    Main intersection results, positivity of intersections, integrability, Hartman-Wintner theorem, local behavior, contact between branches, and singularities of J-holomorphic curves.

    The guiding question is:

    > Why do J-holomorphic curves in four dimensions intersect with positive local multiplicity?

    ## Computational Translation Guide

    | Chapter language | Computational object | Inspection target / check |
| --- | --- | --- |
| branch | local holomorphic parameterization | leading exponent |
| contact order | order of vanishing of branch difference | winding number |
| intersection multiplicity | positive integer local count | winding >= 1 |
| singularity | failure of immersion or multiple branch | delta contribution |
| adjunction | global genus formula corrected by defects | nonnegative correction |


## Standalone Reading Guide

    The final appendix explains a feature that makes four-dimensional J-holomorphic curve theory unusually powerful: local intersections are positive. Two distinct branches cannot cancel by crossing with opposite signs the way arbitrary surfaces might. The local analytic structure forces a positive integer multiplicity, and global formulas such as adjunction collect those local contributions.

The proof uses local normal forms and asymptotic control. Near an isolated intersection or singular point, branches behave like holomorphic leading terms plus higher-order corrections. The contact order is visible as the winding number of the difference between the two branches around a small circle. Because holomorphic leading terms wind positively, the local intersection count is positive.

The notebook models this with two branches whose difference is z^m. As z travels once around a small circle, the difference winds m times around the origin. The figure and check record that multiplicity as a positive integer. This small local picture is the engine behind adjunction inequalities and several symplectic four-manifold applications.

    ## Topics In This Notebook

    - local intersection number and positivity
- integrability and local normal forms
- Hartman-Wintner asymptotics
- contact order between branches
- singularities and the adjunction inequality

    ## Visualization Storyboard

    - A local winding plot shows the difference of two branches rotating a positive number of times.
- A dependency graph connects local positivity to adjunction and four-manifold applications.
- A ledger checks winding number equals contact order for a toy branch pair.


In [ ]:
from pathlib import Path
import csv
import json
import math
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import sympy as sp


def find_book_root(start=None):
    start = (start or Path.cwd()).resolve()
    for base in [start, *start.parents]:
        for candidate in [base, base / "J-Holomorphic-Curves-and-Symplectic-Topology"]:
            if (candidate / "AGENTS.md").exists() and (candidate / "utils").exists():
                return candidate
    raise RuntimeError("JHCST book root not found")


BOOK_ROOT = find_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifact, display_artifact, save_json, save_matplotlib

UNIT = 'appendix-e'
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / UNIT
FIG_DIR = ARTIFACT_ROOT / "figures"
CHECK_DIR = ARTIFACT_ROOT / "checks"
TABLE_DIR = ARTIFACT_ROOT / "tables"
HTML_DIR = ARTIFACT_ROOT / "html"
for folder in [FIG_DIR, CHECK_DIR, TABLE_DIR, HTML_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

BOOK_ROOT


## Proof Visual: Dependency Map

The graph below is a compact proof-state diagram. Read an arrow as "this idea must be under control before the next one can be used." The point is not to replace the analysis with a graph, but to keep the logical dependencies inspectable while the chapter moves between local equations, moduli spaces, compactness, and algebraic counts.


In [ ]:
CONCEPT_NODES = ['branch', 'contact order', 'intersection multiplicity', 'singularity', 'adjunction']
CONCEPT_EDGES = [('branch', 'contact order'), ('contact order', 'intersection multiplicity'), ('intersection multiplicity', 'singularity'), ('singularity', 'adjunction')]

G = nx.DiGraph()
G.add_nodes_from(CONCEPT_NODES)
G.add_edges_from(CONCEPT_EDGES)
pos = nx.spring_layout(G, seed=41, k=1.35)

fig, ax = plt.subplots(figsize=(9.5, 5.8))
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=18, width=1.8, edge_color="#435466")
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=2300, node_color="#eaf2f8", edgecolors="#1f567d", linewidths=1.5)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8.5, font_weight="bold")
ax.set_title('Proof dependency map: Singularities and Intersections')
ax.set_axis_off()
graph_path = save_matplotlib(fig, UNIT, "figures", "proof-dependency-map.png")
plt.close(fig)

graph_check = {
    "nodes": len(CONCEPT_NODES),
    "edges": len(CONCEPT_EDGES),
    "is_directed_acyclic_graph": nx.is_directed_acyclic_graph(G),
    "source_span": '653-694',
    "passed": len(CONCEPT_NODES) >= 5 and nx.is_directed_acyclic_graph(G),
}
graph_json = save_json(graph_check, UNIT, "checks", "proof-dependency-map.json")
display_artifact(graph_path, width=780)
graph_check


## Executable Model

This section builds a small model for one core mechanism in Singularities and Intersections. The model is intentionally finite and inspectable: it creates an artifact, records a JSON check, and gives a learner a parameter to perturb in the applied lab.


In [ ]:
m = 3
theta = np.linspace(0, 2 * np.pi, 400)
z = np.exp(1j * theta)
difference = z**m
angles = np.unwrap(np.angle(difference))
winding = int(round((angles[-1] - angles[0]) / (2 * np.pi)))

fig, axes = plt.subplots(1, 2, figsize=(10.2, 4.4), constrained_layout=True)
axes[0].plot(np.real(z), np.imag(z), label="domain circle")
axes[0].set_aspect("equal")
axes[0].set_title("Small loop around an intersection")
axes[0].grid(True, alpha=0.25)
axes[1].plot(np.real(difference), np.imag(difference), color="#d95f02", label="branch difference z^m")
axes[1].set_aspect("equal")
axes[1].set_title("Positive winding of branch difference")
axes[1].grid(True, alpha=0.25)
for ax in axes:
    ax.legend()
fig_path = save_matplotlib(fig, UNIT, "figures", "positive-intersection-winding.png")
plt.close(fig)

check = {
    "contact_order": m,
    "computed_winding_number": winding,
    "local_intersection_multiplicity": winding,
    "positive": winding > 0,
    "passed": bool(winding == m and winding > 0),
}
check_path = save_json(check, UNIT, "checks", "intersection-winding-checks.json")
display_artifact(fig_path, width=860)
check


## Invariant Ledger

The ledger records the chapter vocabulary as computational objects plus explicit checks. It is a small source map inside the notebook: every row names what should be inspected when the figure or experiment is changed.


In [ ]:
ledger_rows = [{'item': 'branch', 'computational_object': 'local holomorphic parameterization', 'check': 'leading exponent'}, {'item': 'contact order', 'computational_object': 'order of vanishing of branch difference', 'check': 'winding number'}, {'item': 'intersection multiplicity', 'computational_object': 'positive integer local count', 'check': 'winding >= 1'}, {'item': 'singularity', 'computational_object': 'failure of immersion or multiple branch', 'check': 'delta contribution'}, {'item': 'adjunction', 'computational_object': 'global genus formula corrected by defects', 'check': 'nonnegative correction'}]
table_path = TABLE_DIR / "invariant-ledger.csv"
with table_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["item", "computational_object", "check"])
    writer.writeheader()
    writer.writerows(ledger_rows)

ledger_check = {
    "row_count": len(ledger_rows),
    "items": [row["item"] for row in ledger_rows],
    "has_source_specific_checks": all(row["check"] for row in ledger_rows),
    "passed": len(ledger_rows) >= 5 and all(row["check"] for row in ledger_rows),
}
ledger_json = save_json(ledger_check, UNIT, "checks", "invariant-ledger.json")
display_artifact(table_path)
ledger_check


## Applied Lab

Change the exponent m. The winding number and intersection multiplicity should change together and remain positive for holomorphic leading terms.

The intended workflow is to change one parameter, rerun the executable model, and then inspect both the figure and JSON check. If the visual impression and the invariant check disagree, trust the check first and then ask what the visualization is hiding.


## Takeaways

    - Intersection positivity is a local analytic theorem with global topological force.
- Contact order is detected by winding of branch differences.
- Adjunction converts singularity and intersection defects into genus constraints.

    ## Sanity Checks

    The final cell asserts that the generated figures, ledgers, and JSON checks exist, are nonempty, and report successful chapter-specific invariants.


In [ ]:
expected = [
    FIG_DIR / "proof-dependency-map.png",
    FIG_DIR / 'positive-intersection-winding.png',
    CHECK_DIR / "proof-dependency-map.json",
    CHECK_DIR / 'intersection-winding-checks.json',
    CHECK_DIR / "invariant-ledger.json",
    TABLE_DIR / "invariant-ledger.csv",
]
for path in expected:
    min_bytes = 80 if path.suffix == ".csv" else 512
    assert_artifact(path, min_bytes=min_bytes)

for path in [CHECK_DIR / "proof-dependency-map.json", CHECK_DIR / 'intersection-winding-checks.json', CHECK_DIR / "invariant-ledger.json"]:
    data = json.loads(path.read_text(encoding="utf-8"))
    assert data.get("passed") is True, path

print(f"Validated {len(expected)} artifacts for {UNIT}")
